# LSTM + PCA Classification — POC-ABS Features

Reusable clip-level LSTM workflow. External test = 1 clip per emotion.


In [ ]:
import sys, os
from pathlib import Path

ROOT = Path(os.getcwd()).resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent.parent
elif ROOT.name == "comparasion":
    ROOT = ROOT.parent
elif ROOT.name in {"lstm", "svm", "knn"}:
    ROOT = ROOT.parent.parent.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Project root: {ROOT}")

In [ ]:
import random

import numpy as np
import pandas as pd
import torch
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report, f1_score, recall_score
from sklearn.preprocessing import LabelEncoder, StandardScaler

from comparasion.core.models import (
    build_clip_sequences,
    evaluate_lstm_external,
    fit_lstm_split,
    split_external_clips_by_emotion,
    split_main_sequences,
)

In [ ]:
FEATURE_PATH = Path("comparasion/output_casme2/features/poc_abs_flatten.xlsx")
SEED = 42
EPOCHS = 30
BATCH_SIZE = 16
HIDDEN_SIZE = 96
NUM_LAYERS = 1
DROPOUT = 0.0
LR = 1e-3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Device: {DEVICE}")

In [ ]:
df = pd.read_excel(FEATURE_PATH)
META_COLS = ["emotion", "subject", "clip", "frame", "label"]
feat_cols = [c for c in df.columns if c not in META_COLS]

print(f"Shape: {df.shape}")
print(f"Features: {len(feat_cols)}")
print(df['label'].value_counts())

In [ ]:
le = LabelEncoder()
df['label_enc'] = le.fit_transform(df['label'])

X_all = df[feat_cols].to_numpy(dtype=np.float32)
scaler_tmp = StandardScaler()
X_scaled = scaler_tmp.fit_transform(X_all)
pca_full = PCA()
pca_full.fit(X_scaled)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n_90 = np.argmax(cumvar >= 0.90) + 1
n_95 = np.argmax(cumvar >= 0.95) + 1
N_COMPONENTS = int(n_90)

print(f"90% variance: {n_90} komponen")
print(f"95% variance: {n_95} komponen")
print(f"Use n_components: {N_COMPONENTS}")

In [ ]:
sequences = build_clip_sequences(df, feature_columns=feat_cols)
seq_main, seq_ext = split_external_clips_by_emotion(sequences, n_per_emotion=1, seed=SEED)

print(f"Total sequences: {len(sequences)}")
print(f"Main sequences: {len(seq_main)}")
print(f"External sequences: {len(seq_ext)}")
print('External clips:')
for item in seq_ext:
    print(f"  {item.emotion}: {item.clip_id}")

In [ ]:
results = []
for split_name, test_size in [('60/40', 0.4), ('70/30', 0.3), ('80/20', 0.2), ('90/10', 0.1)]:
    seq_train, seq_val, dropped_classes = split_main_sequences(seq_main, test_size=test_size, seed=SEED)
    if len(dropped_classes) > 0:
        dropped_names = le.inverse_transform(dropped_classes.astype(int))
        print(f"Dropped classes for split {split_name}: {list(dropped_names)}")

    model, transform, y_tr_true, y_tr_pred, y_val_true, y_val_pred = fit_lstm_split(
        seq_train,
        seq_val,
        hidden_size=HIDDEN_SIZE,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        lr=LR,
        device=DEVICE,
        num_classes=len(le.classes_),
        n_components=N_COMPONENTS,
    )

    train_acc = accuracy_score(y_tr_true, y_tr_pred)
    val_acc = accuracy_score(y_val_true, y_val_pred)
    train_f1 = f1_score(y_tr_true, y_tr_pred, average='macro')
    val_f1 = f1_score(y_val_true, y_val_pred, average='macro')
    train_uar = recall_score(y_tr_true, y_tr_pred, average='macro')
    val_uar = recall_score(y_val_true, y_val_pred, average='macro')

    results.append({
        'split': split_name,
        'train_acc': train_acc,
        'val_acc': val_acc,
        'train_f1': train_f1,
        'val_f1': val_f1,
        'train_uar': train_uar,
        'val_uar': val_uar,
        'model': model,
        'transform': transform,
        'y_val': y_val_true,
        'y_val_pred': y_val_pred,
    })

    print(f'### Split {split_name}')
    print(f"Train: {len(seq_train)} seq, Val: {len(seq_val)} seq")
    print(f"Train — Acc: {train_acc:.4f}  UF1: {train_f1:.4f}  UAR: {train_uar:.4f}")
    print(f"Val   — Acc: {val_acc:.4f}  UF1: {val_f1:.4f}  UAR: {val_uar:.4f}")
    print(classification_report(y_val_true, y_val_pred, target_names=le.classes_))

In [ ]:
if seq_ext:
    best_row = max(results, key=lambda row: row['val_f1'])
    y_ext_true, y_ext_pred = evaluate_lstm_external(
        best_row['model'],
        best_row['transform'],
        seq_ext,
        batch_size=BATCH_SIZE,
        device=DEVICE,
    )

    ext_acc = accuracy_score(y_ext_true, y_ext_pred)
    ext_f1 = f1_score(y_ext_true, y_ext_pred, average='macro')
    ext_uar = recall_score(y_ext_true, y_ext_pred, average='macro')

    print(f"Best split for external: {best_row['split']}")
    print(f"External — Acc: {ext_acc:.4f}  UF1: {ext_f1:.4f}  UAR: {ext_uar:.4f}")
    print(classification_report(y_ext_true, y_ext_pred, target_names=le.classes_))

In [ ]:
comparison = pd.DataFrame([{
    'Split': r['split'],
    'Train Acc': f"{r['train_acc']:.4f}",
    'Val Acc': f"{r['val_acc']:.4f}",
    'Train UF1': f"{r['train_f1']:.4f}",
    'Val UF1': f"{r['val_f1']:.4f}",
    'Train UAR': f"{r['train_uar']:.4f}",
    'Val UAR': f"{r['val_uar']:.4f}",
} for r in results])

print(comparison.to_string(index=False))
comparison